# AsturCamín · Afinado del modelo de decisión Laya (2× T4)

Adaptación del notebook oficial de Laya (`laya_finetune_typed_decisions_2xT4_kaggle`) para entrenar
con los avisos etiquetados de AsturCamín. Los datos van incrustados en este notebook: no hace falta subir nada.

**Antes de ejecutar**, en el panel derecho (Settings / Notebook options):
1. **Accelerator → GPU T4 x2**
2. **Internet → On**

Después: **Run All**. Al terminar, descarga `laya_asturcamin.zip` desde el panel **Output**.


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")


## 2. Install Dependencies


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 2b. Datos de AsturCamín

In [ ]:
# Datos de entrenamiento de AsturCamín (generados con modelo/exportar_finetune.py)
import base64, gzip, os
os.makedirs("/kaggle/working/datos", exist_ok=True)
FICHEROS = {
    "finetune_train.jsonl": "...",
    "finetune_val.jsonl": "H4sIAMIt0WoC/+3dS3PbVpbA8f18CsRriUWAT7kXXYrbSadKTjJxx5vx1NQVCUt0QEANkOyWu/JhsuyFq2aqd9lM1eiLzTkX4AMURRIASYnyf5PIfILkveeeX537+MeLQf/FS+fF+bvTer3ebLw4cV78LYp/+RBEf9PbTTIaxz0zHIR6TzIyI19vfv/ifDJIIqc3GPdN34TRS+fP5tYZh07oj4d3v40GvcjxQ2c4GJm+0/cDp2fieBA4NyYx/ch5awKTvH8fXoyvTPzS+f603WieOJO7z8YJTW8QhSaQZzmjwWgcmHgg7+G8ljfvRyfOJ7lT7xtG4cjc/Y+RV3kTDf1wJNcgN/flNeSV/FguSu76y8Af3sg9QTCeDIxektd0rp16re4MhyeOsZ9i6I/8KI6Cu39d6XWHg/BqHMqrjOS58kLyDRjH9Wqe83///Upe89Ug7sll9QZ3/wr1FQNjr/ylMwhHfpjYqzXBSK/sz4NkFMWD9NPI4/TiXzrJIHSup/f8wYkHfnIV6SM+GHlfuZz+QK/9nX4dvSiWzy1PCqM/yJfY90OTONFlMLgy9vlJes8g7A3kvp58xujDoKfv2PPlh0vvtR8zkS8/9uVh8ld2X/39C/1Z/zr2k5F86Yn+tP94/yKKr0w4SIbRe7lB/j26vfH1z/cvetfRoCd/n8jfgzAZxeOefWJ67//977+P7/7pzJ4uH+nSd67SFzexI3/46bX80Xktn8B3/joe3P0zlN/ZD/Ufcm9vNJZHypUaJ/aTKJBf0pHmcxPLZ/aH8qOE0fRZQeRcy1evX2QtvaRePBj58qVk1/1KrsyPJ9kvJd+vNkL5rWOTXfCfpAHJtxHNXj7Jfk/59418lbPHOx/G+l/7Eok0kXF8aRv9peld+8mJk0Q9M4lC/bPvJzexH/YHw4E2ysS5dWK5V+65+y2+jAJfvn1z91l+BXmaLw1Yb4nkleSt5MKcvjbqvj7+0gSJ3iDfytVYP/nAn/jy0OuBH0hHkO94aJ88lGvTN0ovvWeCT/L87Bt5PfTjK9suso+s3VQadRJpO7r2tW/p28tzb3z5SWJ5YWlLev2RdqPru8/S1iPHjGJzoz03sh9Qr7tvHxCPpEHZ/qhNKuxJm02mL5Xop4j0ohLpyNMO83f94eW37UkskXtNIj9g2nIT+Ugj6e7Tiz+/HUsfT7/H9Op/XPyd5JPqx7+1vS9Jf5Pph/HlUenXPxyH0h9u9JuSnudLcPogfV0uzATj4WVsP4bp6VevN8lH0CsIAjP7eQafsitP3yC7tr/E8kISLZavS5vIYny4lVe5GqchTBt47Evr9aWByF9XGtzkeieDq4H+Je18lL3sy8XvfqJt2n73pme7uXxxNoboz7f8w5842mRDDaLaHKUr6lOTsR+M5JP2orAvfTaK9UmJvJL+enKj/Lyx/uLR5Ud/FGUtVD+MhPTZlWQf/Y3GJ8cML/WHyQLD+Qe/l35giTz6xcfST0MbOoOX02aSaOedtg39CdJ4l5hgYj76WWu016i/ejjvt/HdZ9tNe+PQH+kvI81n0B/LbUZ+kn6kHzFxPkprkeuaxmO52l9/1etN/Il95f69cCZdOd4UzWbPzqKUvPo82P7R+Ytvx4DeWD600V9Qr97/u45GYSTB4SQberRxybdg8oONftqxRCj5bPILDBe+HenzsZmYMN97VsS5/3j/4mvzUYL59/Ic+RXkLexXcK8VOjdjX77Lm0B6mTQxE9ec1x9lhAuiJAtk8u7SRCUkpj3IMXFP4uy0H0gbssOLBoBeFET6pSy0CLmC8/TtJXjrADS07fT+tWjLzT6mfJM62MlItngt2dvZNrgQeReb+KwfOxKVrrSByxVrW7IxQb/LULpZMAskMhq/dH7M3jT25dUlBmp3ijSpyH8REnSnMXf+qtNxIPGlDfY1Rj14EU5P2pGv7WDWm7PLeCXtWDIjuZQfLpPR3W+2dy913snCoO/M2kPWGuQtbqTr5hqW7RbShBY/hA44TtQb32jnsJdks697g9Nylzm5NzTIlf+n7URpkvJf8obBvW4URuPg4V70dnwjoUgH/8Weo1eVJT6zRpAO+xofQx0l00YT2o+cXeAfpU/r+4x1UEtjT4Er0bFPY7DmafHEDwex/tCztz+xv4VmpHqzpADynQ+Nfe/Y10tL26+9yGgkP0p/4Yrk/W40BEZh4YvqaeInLdRem3wto7t/9kJNRCfm1mg8Ta9Jv73pe+g/5UOkyYFNmfQ6ftVk7ioK+g/kcYG59IP0fefDl16Xpj/mchAMJMQl2yVPkkGvyC+yW5cH7uzmxTHTTW+6P5bIY1eH7dnVuyfz0D17oVUfob7w3u7CY72F2xu597zfyhe+M/mRkmy0SH/S2YusenP5of2Fh2RPttfw64o2vL/3udcy9/NWv7749d/+kdNk1yuvyWliKa1f8iSNDdoPTJqTBIsROR2qfhk6Xqc2F+X521PXnZJSXmElKX+MNR7dbM/Km1hSbrXlw7L03FqnIC0b8oztZXkpg/2DsjSBvPw8S8SW2BJbYktsiS2xJbbElthy/7ZcjuGPJMz6Q8J0EebhhJk+efGd3M3v5C69U30lMF3vrDwwe9cykhhHumEoLS8XKNOYJb1mbAcHi0uhZKC20egoOahkWf10HLP5TRRoV9LaZqwR5YOp1RYsevH9aTcLvoEEz+oMHYQTuYB19U3Pq7kPK1Qjj6aheYa2ao1dMXQp34ChMBSGwlAYCkNhKAyFoTB0/wxdmxqVNKm7L5Nurno2ciZtFDBp/QGTuluZdHeAW0vS3b7NY9U83YZbnqSvZThKtJtEGknuftNqnqSW/kQTtn5+xJqXPbs5a56ftttZhDXjUaTBc+U82pKu7NSaBYubzVp9V9NmNxY3xaSgElSCSlAJKkElqASVoBJUgkoKncc9ldZt1cuz8tzmu5qJRhIVgyzeLsbOWyftEVl8lGRDRj1JmUPNhxd1uf06zSVfZq8cjaIdr8o8q3V3xUs7PMuFSsJFwRJbYktsiS2xJbbEltgSW2JLbPksbdmuMIv2h4+R7SHXZpqeaGi2oc7kl2la3Wj7/HbwUVrqSRoKU1dloTC3evO779aXMeeTZleWMncvzfrupLlh/x/KmFATakJNqAk1oSbUhJpQE2oyN/YAc2P3uVqz091fDXPVkk2v9VV+PyDvEbYDOqu1ni40KWoiTaSJNJEm0kSaSBNpIk2kSVHz+Iuantcpj82fNVXIRqLF0uZttixTzzWxTWfhXBNpoxcmvIp9TfKWLBre/T7UQcDtLIDUaFjoT8cbbfBKUdvgs9icZoHLSM0175IodUWYBVHaKLQ5EMs4USkqRaWoFJWiUlSKSlHpk1Jpfmg9xAa1LgXPL20zIK/KZkBlECq3vdHkSjKELLIFaTRecGc6QlwY551/ZapJc9Mhm43muuLn6k1o3a48h2m2MBNmwkyYCTNhJsyEmTATZsJMmLmamRVqnW/TlHruxcsoVuDNUamnmkh4+2lwKSxKfCs2P7dTrS7u7EurSuPmB5Obdeu1G1tPuy2lzPm2PaZIRdPzal4BaNrghTSRJtJEmkgTaSJNpIk0kSbTbPcxzdbLwdMrCU+38DRbFnRuv6DTazbKu1O7rnSNjzLeB3efs5TXFjUlY+wvzayVDqvrN93O/NSTplvLL+50O7te3bn5NJTiuwi1C1U3K6GTxZ2oE3WiTtSJOlEn6kSdqBN1os6jL3c2m+XZeV2QnSZZZufjq9Pr1DpFp9U2a24Bd15KJvD02enCTtgJO2En7ISdsBN2wk7YCTth557Y2XrMauee2Kl76tqALDmkY/EY6p/7KX7Wa2eHKn7q1OSDINQDoSAUhIJQEApCQSgIBaEgFISC0P0gtNWujNDQH0vQHWnvWAXQFQepuO2vDrukszgtvUahQ1PW23Ipj2A9J7pEl+gSXaJLdIku0SW63Lsu58PXIXYNWkFJlzNSnvsZKd0KE2nn2c8kkqRMolh+HJjGYwsZvTU0dpBNx7hr+YSxPQhEmpWjCVMclT2r80DCdM8KnYDCfkH4El/iS3yJL/ElvsSX+PIp+TKXF+yUmO4hqpWPtTHtoYj5TMqV3W55YS7vMStalFTSn2iC1l8YoXTwzSHzq7KS3HKF5hYHbJYoYO6wfsnGQPASXsJLeAkv4SW8hJfwksmxVbhJRfMoKpqNpld5a6A0i5bBOM4i7vSUzQfqmvH8XBQdFxKbZSkKJTM5sad1WpimIVXyExkoJwrV/s6OSclee/OKzW6J4zgFsjs6jVOrveE8w8SluBSX4lJciktxKS7FpbgUl3I+57M7JqXRrLBm852RFDrNYjS/TCPC+Wl31TLNTlO9qcd1ClrHQhaJJRoxkwVonnazUGvGo0ij6BIyXyejp7Vn0Klb6JjOtQKdnxQaPeKOQeATfIJP8Ak+wSf4BJ/gkzWdT39NJ9I8gvm2jW6F+ufXJuhr9jibYztLzG2mNj0QRbtw3/7rbe96HHxaX+Os1RaO6VzxXBt50xxvmaK59ryy4hmNoh2v8Wzt8HjOjfVOVnkiTsSJOBEn4kSciBNxIs4dizM/tB5Cne5O65v1nDrrG9TpbqFO5t3uxZ0VdhK6sLFgZGzea3OHa2OrmBLTbu2s3DQjNr3xUAJF36yqfTY6X+3vYM6tJ9iKH92iM2xP67X2Ez0VJbn7DDkhJ+SEnJATckJOyAk5ISczbKl7Pq26Z7NeL+/PnyR+RLa930R9kw1Ql1Gs+/1MxwaJ/HL7G02vJEfIYluQxuP1BVCNMR8WtyRKX03Q+86/MqUKoA/Ov91QD23UOkXroV6hPW/XTr5ly1tkikyRKTJFpsgUmSJTZEoxlE2Inl0xtFnvlMfoj2K5QBq9hNihGQ0ms7QkiS7jpS2I0gHrl6Hj1Xd6dsoW+9t6nXWFztUb3LryFObWwkk4CSfhJJyEk3ASTsJJCp1PotDp5XTplSx0uoV1SaGzkC27lTe4teNUT/KzaLHWeZJWKzVG+veqlT/H6aGapRZrbncgp1vrFi1Odgst1qQ4iSbRJJpEk2gSTaJJNIkmKU6yUvMLK056ZxU2o51lQJNIEjOJZPmxYBqTs8WbqgSbCdsNfyJpeZrapYPetXzc2JbxpI05mkFporv9aSneoyzv1L1oiyG1UWuyey1CRagIFaEiVISKUBEqQj1Goebygp0C1X3OCzkPBdRnUuBsVFjJOang02Q7n644XeXeCs9H82mzcA21sbvDVTZOyT0YUV2IClEhKkSFqBAVokJUiApRISpE3RVR3fJE/TlM1agj17WZZiQaCzRN8mPJRwNVQDoypJJcV099ClXRRuG1oPLFHm4p6MHc6eFO3Ik7cSfuxJ24E3fiTtyJO3HnrtzZqDJ1N00tpf1LpmTS7WsDk2Yli8XRB4qcuXM8z0/b7Sy2mvEo0rC55MzXyWijMR/cxnYzOYuf7Nna3VpR5uGCTbAJNsEm2ASbYBNsgs1DYnM5hh9irWgRb7psZHu4taK7o+19b3YqTMV9LUNQol0j0uhx95uW6SSd9CeapPXzo9Tj1DfTbYn2Yc1moUNTLiUDKLsvkUY/uAk34SbchJtwE27CTbgJN9nmFnse+T5F3WZ5fH798BCVDQ8X359202Am3WlsRw17koq7UOXUx2RRNpAoWd2am7bALS5Nt7E7am6cRws2wSbYBJtgE2yCTbAJNsEm2GRT3OeAzVZ5bP553akqK425eNRKbl7tcYiz++yKm2wYhDfxJt7Em3gTb+JNvIk38Sbe3J8323hzS282a+2DVTgXVnayYRDuxJ24E3fiTtyJO3En7sSduPPI3dl5LHceFzrbhU7tPI4iZwNsgk2wCTbBJtgEm2ATbIJNsAk294TNVv2swna1GnUT6SoyeMr/05hrFrIfCfzfSuSQjEN0EcWfbPj+YZIe4ZlFuiCLzsvnc656ehaM07RvmaS59nwggrZq7q42rN1AUOHr7gSa3H1mXScEhaAQFIJCUAgKQSEoBM0PrYfYsdbFnF+gOasczfmDzUVm6XKajeiAEF9FmgfkQ+fcm8uw/DlOTVjBkzex5NqKyjWnbta84ks3m7si5captDtFJXsFYUpMiSkxJabElJgSU2LKxeHrkU5AcdmF9tl7ssIutBc2DoyMzXlt3nBtpked3DrSj9Nk2PTGQ4kRfbNqDq23r/WaWfzdfPyJd7audqnBRTPNvDRP6wfbJKjgeZu2PAk0gSbQBJpAE2gCTaAJNIHmFzR/tpGzZ6OAPR+qZbpb2XN3R1Wupedu3+bR5NltlJenxOW+JpSz8zZnubpN3uYDhIT/D7F09J7/QdpKaJOYKJtEu6zR8O73oY4DzcZywfMbfYlX2UuULXxmcToaRes0quedFKx6euLX7S1qwxtlTzSKRtEoGkWjaBSNolE0ylRaptJ+SaXPdr1bBaA2JdZ2N8+h+pJoaWgcaa4snUWa+42fjVe/DJ26jbtB2pdsyOmZWwl4Ua4Eev721HWbWbCVPEx71P63EPLqxYug3QPOtuVoTtyJO3En7sSduBN34k7cSRV0LUO9HEO9klVQtzBDD1QFPaBCd/eJ7iO04ZVH6Dd28yAZDSR49aerN4124r4N12971+Pgk97+7eCjNM/lfYPm6Lz3pP0WOUvsFuTVWiztxJpYE2tiTayJNbEm1sSa1Dgfp8bJ8s7jqHE2K/Dyazsm+WEUD6dVzDSO5abXvjGh5DbOhUQKP92k9o3mV8m97WjzD6wIzEc/EmX9NNqlZAJiQkyICTEhJsSEmBATYkJMiEn98hnUL5uNxwLmiQx5affIgqVkHjIESv4c2uT4yPjZKHQcynp+HvI8lHX4dMEn+ASf4BN8gk/wCT7BJ/gEn+BzV/issHmtdladBno1/qihWvpi2Pfvn4AytaZk2APpppM0rmnQlkvVgLoOoY9q0HbtrKBBvbo85bkZ1MOgGBSDYlAMikExKAbFoBgUg2LQXRm0VbUAOhzfak/UZZw2VA4lQezbHWmzauiMoUdX1OzWGs8OlA1ACSgBJaAElIASUAJKQAkoASWg3BUoz8qD8rWMP4n2i0hDx91vutdNNDskZTpEXUaxbgzrryx1LhPzwjjv/Cvz5DYE6hY6gfM4aMliTWgJLaEltISW0BJaQktoyZkn7Ae0u/2A2hWqlWkuoumyzR1sMqLjQXwVaRqwNHf2xiQaH+282jQPk7xiMPSXjjrxDn3SSQlqtmrtQ1Fzt+ecJHefwSbYBJtgE2yCTbAJNsEm2JwPX4eA5opTTVw2nn3u0OzWy0PzwqRJWVbDjKPRQn0yd1jmTkqUqRs3VCibxdnoFtrRZ+2RJRQoMSNmxIyYETNiRsyIGTEjBUoKlM/PjWfdKsspg77mkLMJr7P0POVkOiYsnHUp/9IWq06c2ezChFexH211Vuby86sJdPeVy4Ib+nBqJggFoSAUhIJQEApCQSgIBaEg9EtDaMetV1qCKTnJYozSaa4t5ZGx4VWC2XhoZHCzw5EjKcx05Ppl6HhdDRwfzFf5WbKtrWfJLspSHmds1JVEUdOwoQn1T4mXsSThys2Hsek2a17xgmf3cCdo7namLN7Em3gTb+JNvIk38SbexJv3Y/gjTZd9UJwu02WfxZ4/HdcrD863vib4fWluPbvHT1qLW4zPs0QzC6MqSrezgE43RWfenG5n10szt2Fnu9YsvjyzxfJM1Ik6USfqRJ2oE3WiTtR5tOpcmxqVJKi7L4J+8bvO7vZtHq3k6XUOKdDcus352k4bmIO0s6VrP3vmVkJitJ8ln5sm3Dbb62qgdqiVcD/QLDUH0natszOQMu0WkAJSQApIASkgBaSAFJAy7RaBPkOBVlj5eS2dVWNvPpblxpZZYJsuCU1OPa+zJE4zBeeKgqg8etcF0Q17DrV1B6GHi6ESWsKP9+fgFimG6qTkp1ELRZ7IE3kiT+SJPJEn8kSeyPPZlUIbOYg2SkJ0cTau+2VCdJ9zcRvN8g49D640LGjCPxrENtbKzy352zz9sblBNlBkE3FtYJOuNbYjiJ2S60pTuXXSnpPFUUlKZHSU1Dq0eXPZubqrz+cchBNJ99ZhdL1FV03MPa3v8NyUjXVQOApH4SgchaNwFI7CUTgKR1kPyg5Ex7kDUatdHqE/hxoKP0VZijuLldNoPFv32e6epPHOjsBpcXSp7vn9abvRzOKrpharbPk6Gd135d62tHVr3qFWezK1FlEiSkSJKBElokSUiBJRUuBkpu2xz7RtdSptb2sS7SZRtkLTBBI6pgeuTEesyyhWAy4OFnrPt4OP9pQUSX2NJJmSEwpkdNWnZu/LSzxfRXGah1RZ4rmPuuaZoJVzPQEoAAWgABSAAlAACkABKGs7H2FtJxvcHklBs1vJnJM0TM3n1dp8IR0fzKIy0/HBli1T3Ogtb42kcVWLmlVOUyle53QbhSbPVju6k7mzQBNoAk2gCTSBJtAEmkCTSidLOY+/0nl27OoscnhKcWV2C+1UW2W3IEqZCBNhIkyEiTARJsJEmAgTYTKX9tiF2amXF+Y7/3pg4+UkkiRNolp+XMja4Uk2yF3LJ4ptLU93GNKMKV48F+X8tJtFWjMeRRpEt7flhn1oS1Qvm7UGqzSRJbJElsgSWSJLZIkskeWRyjKXF+x0jqz7nGuVh5oj+0wo2W1WOoIzXY4pvWJo45iMSHr2iG+7hEnTk9w2QCt2nW3Uaksby7bK7iu7x6Jlq+YeypYLZ3xCTIgJMSEmxISYEBNiQkyICTGZDntEwuzWvfLCtEFraBteTxPEWMLwjSROOhDcfZYY0V8InYEdHSb+VXoGpuZXms0tiTO8+32oUb+bd2f6/AvjvJPnV9v+Z/f7z7rdQrv/VDzUhNom8ASewBN4Ak/gCTyBJ/AEnsySPa4jNbv1Rnl3Zgn1NEDN95adHVli42Z4FfvSb2ePsqH3xI5d8+1qJZ5KE5OoF321zM0/RWkCWEWbG6bRntXOimKzvsNSJ9gEm2ATbIJNsAk2wSbYBJvsNos2n582W9XOztyuynn+9tRrN+a7/VxIzPCT/PRZecDe58+6rTJ7yzZZnQkqQSWoBJWgElSCSlAJKqlgUsHElCtN6VWoYPpbHZmZDlD2YExNY4xYT08YSfS0zLRzZKFS8g4ZACV7DjU1zi3YPG23S+wBlL1ulU2AFtZJ3tsIyDtcBZNzTNAm2kSbaBNtok20iTbRJrvMcn7m8Z+f2fXa5QH6JhrNE6DsuJKL6YzZhxdlnqR7CCmWNAKl7szYqqHlg5nT02hQmB2Eos1d2Wmb+1Nbwem5hfamXXuy5lLqQfkTkAJSQApIASkgBaSAFJBS/mTnoOPfOahZgZ8/+ZeanGuriyb+R4lIElY/2XiWzqP97rtp2fOHycDXLE4HFskEBn5uOu13362vbj44mXa7Cue8iGke8KTGEE0ol2qc3u5ASYkTUSJKRIkoESWiRJSIElEeXJTz4esQKzRXcNKFk8+ck53ynIzXctIkp4PBlJPRMieXzzcpJ0rpdcaGWckMHavCUP/cCMyGOLHgSk2v0ErNtbTcsFDzcLB0gSWwBJbAElgCS2AJLIElsASWwHIXsOyWh6V21J4M7UGwkN1MQ/BSgXLdmsxHFKZXaxedDVvfXfHyyQjTQ5gIE2EiTISJMBEmwkSYCBNhIswdCLNVYXfZN5JIXqdRuOfLUOdITwzlX7lYOR8TJOJ/E0vffuV/kBYT2rxlCtCHlm2e3TvZJPcS1RZjbrEdbYkTNd1ai91oESgCRaAIFIEiUASKQBEoR5xwxAkKfUCh7Qp1zhkus91onSiROKeDvyQRH8Y2wbED7kIed2sLoKHkfAIViSAaJ5M9SnMP2/60C52lWW2VJs7EmTgTZ+JMnIkzcSbOxJk4M/cJ6jln1jc4092i2snGs3uR5ll5ab7TMJtI35DRUv6fBlmzkO4so/G4RNmpdXYlyoUDVB5Rk6zMRJNoEk2iSTSJJtEkmkSTaBJN7k6TnQr7yH4z9q9m8dGO/NkwcP721HU72SD1y9DxzvK7/MidWSCVHEt7S/DwKswTCzKbwGraISNPOU0W3+jHPdw5mQvapEqJK3ElrsSVuBJX4kpciSs5nIQlmcc0GbZbr3A4icmyvNk4lI0B35+2G/UUMHrLjzL2STaQ20DWPiQLpZpFrJLl62R0/zySvVUk64UqkiylBI/gETyCR/AIHsEjeASPTwmPa1Ojkph0K+7vQ43y2dcou43ynHwtA1KiHSWarq4MJHgE/kRTtv50zLqMYlXg4nCxPPP1p4G0oqe4iLILMAEmwASYABNgAkyACTABJrNemfWKKB8WZRNRPjTxdYfb8jwVULKMElACSkAJKAEloASUgBJQHgco3Rwo3QLzXV1AeUhQVjiExC8Lygd2gc1NiX3SxctmrXl8+/Ukd5856BJpIk2kiTSRJtJEmkgTaSJNpHkQaba/kNLlXHSmECobtcYx1i/XsrIBK2ElrISVsBJWwkpYCSthJayElbtjZWffrEwHqYV9YE+7WRQ141GkAXKbnXqykBqNot3WIs9qZwc7jFLDGCspcSNuxI24ETfiRtyIG3EjW/XAyONm5Jn7GAsr9V9vNOfSDE9vl9cY2zElcMK734c6Enjt5RrmGxNKzuRcSATyP5lqtcxBOJH3XyfSVq1VWKSN57YOE47CUTgKR+EoHIWjcBSOUsakjIk/d+jPboXjLG2nkN43kvia5iIml6LlY+d8jJAR4FsJIZJ6CDOi+JO96wcRoc31HgBpY8UyzfyrVCPpTSzpus6xfdikXrfmPoxSjUX6qZe2B2pyogkuxaW4FJfiUlyKS3EpLsWlj+RSL+dSr+R5mG5hlz678zB39073Udrc2dxaU3VvoD0UQaVjGhuJJXl0LBtD/XMPc3Td1vPbmhZ6Qk/oCT2hJ/SEntATekJPSqKURHdXEm155fUpyUIiA/BfdRT2kzRiX0sXtgpNR6g0rn21sK7z7anrdrIwKhmW9pVgGZM/xhqHbh5c37lF9bLdrJ2VqF4WOS5TFMgyTxAJIkEkiASRIBJEgkgQyTLPfS3zbORM2ShZzlw0pUs5c9flzFaFs04uFoNwusrz2g80Ri7Pk/1l6Kgib8xIMghFwrSEuURNb2tpnli42RxXMxMZnEqu5Wyc1byi8qzXWoeaNos7cSfuxJ24E3fiTtyJO3En7sSdh3DnXiuZ7UaFQzb/rmmijcbZ+KBOzFrPdKwYaQKtPUjZF6ePuRp8lNb68DpOtz73qNGgkK4Mfdu7Hgefqi3h3DCB1j1bt3xz9QzaRiGGMoMWhIJQEApCQSgIBaEgFIQyg5YZtF/YDNp26zE2tZV7Lkx4FfvSp7MQF6RhecXOQX+K0iRwv5vYPuqxKku5BeJEnIgTcSJOxIk4ESfiRJyIE3E+C3F2KlQ63/o2v9dKZJTOBbXzXX0d9D9GMqZJxzx/e+q1G5lIp2FzxQzclo3HQdrHskebW4mEUS03C1derOyCz61Kne3G2uNTJLSEH5dLnW7Ne9SNapO7z+XoaZ+IPbEn9sSe2BN7Yk/siT2ZcsuUW6bcHnapZ+esvENfjfVn11Qt2ywoWIjKOT1+910WRM14FGl8LGZH3SZowzzZdsGypdeu1XeGx40bBVG4BI/gETyCR/AIHsEjeASP4JFjT44cj90K02bPxUzSHjThHw1iG2vl55b8bZ7+2NwgGyjSPWdX7iHUrZXemnbLDYM2+bP4rFmvscPaJfvU4k/8iT/xJ/7En/gTf+LPg/tzOYYfYupsEXK6G8lZz5GzvoGc7hb1SqbO7mbq7P8DwlZ8SFDKBAA=",
}
for nombre, b64 in FICHEROS.items():
    with open(f"/kaggle/working/datos/{nombre}", "wb") as fh:
        fh.write(gzip.decompress(base64.b64decode(b64)))
    n = sum(1 for _ in open(f"/kaggle/working/datos/{nombre}", encoding="utf-8"))
    print(f"{nombre}: {n} avisos")


## 3. Modelo base y preprocesado

In [ ]:
import os, json, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

MODEL_ID = "convaiinnovations/laya"
print(f"Descargando {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)

# Los avisos están en español: se entrena sobre el checkpoint multilingüe si tiene la estructura completa
multi = os.path.join(model_dir, "multilingual")
necesarios = ["tokenizer", "encoder", "rl_agent_config.json", "model.safetensors"]
if all(os.path.exists(os.path.join(multi, n)) for n in necesarios):
    model_dir = multi
    print("Base: checkpoint MULTILINGÜE")
else:
    print("AVISO: el checkpoint multilingüe no tiene la estructura esperada; se usa la base en inglés.")
    print("Contenido de multilingual/:", os.listdir(multi) if os.path.isdir(multi) else "no existe")
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

rows = []
with open("/kaggle/working/datos/finetune_train.jsonl", encoding="utf-8") as fh:
    for linea in fh:
        row = json.loads(linea)
        rows.append({
            "state": json.loads(row["state"]),
            "questions": json.loads(row["questions"]),
            "gold": json.loads(row["gold"]),
        })

items, skipped = items_from_rows(tok, rows, cfg["max_len"], cfg["head_max_len"])
print(f"Preprocesadas {len(items)} preguntas de entrenamiento (descartadas {skipped}) de {len(rows)} avisos.")
torch.save(items, "/kaggle/working/train_items.pt")


## 4. DDP Training Script (`train_ddp.py`)
We write the multi-GPU distributed RLCD training script using pure policy gradients with proper scoring rules and DDP gradient synchronization.


In [ ]:
%%writefile /kaggle/working/train_ddp.py
"""2xT4 DDP training script for the typed-decisions fine-tune.

Training semantics -- item construction, encoding/collation, the RLCD objective,
the sigma schedule, the calibration holdout and the temperature fit -- come from
the canonical primitives in `laya.train` / `laya.common` / `laya.calibrate`.
What intentionally stays notebook-private is the distributed orchestration:
process-group setup, rank/local-rank wiring, the DDP wrapper, equal-rank
sharding and the effective-batch accounting.
"""
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file
from transformers import AutoTokenizer

from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import (
    calibration_records,
    encode_item,
    rlcd_loss,
    save_checkpoint,
    sigma_at,
    split_calibration,
)

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256
    max_len = cfg["max_len"]
    head_max_len = cfg["head_max_len"]

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)

    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)

    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)

    # Held out before sharding: `split_calibration` is seed-deterministic and
    # rank-independent, so every rank withholds the same items, and the shard
    # split happens after -- the calibration slice never reaches a training batch.
    train_items, calib_items = split_calibration(all_items, calib_max=400, calib_frac=0.1, seed=20260922)

    # Trim before sharding so every rank gets the same count (drops at most
    # world_size-1 items). Otherwise ranks can run different numbers of
    # micro-batches and DDP deadlocks on the unmatched all-reduce (#678).
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]

    EPOCHS = 4
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # RLCD baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)

    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()

    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0

        sigma = sigma_at(epoch, EPOCHS, SIGMA_START, SIGMA_END)
        steps_per_epoch = math.ceil(len(my_items) / MICRO_BATCH)

        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            encoded = [encode_item(tok, it, max_len, head_max_len) for it in chunk]
            batch = collate_items([encoded], tok.pad_token_id)

            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )

            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)

            # Canonical objective: RLCD policy gradient over noisy logits + full
            # soft cross-entropy guidance (`laya.train.rlcd_loss`).
            loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                             samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)

            window_start = (accum_step // GRAD_ACCUM) * GRAD_ACCUM
            window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
            scaled = loss / window_size
            scaler.scale(scaled).backward()
            accum_step += 1

            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item()
            n_batches += 1

            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item():.4f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            save_checkpoint(model, tok, dict(cfg, max_len=max_len, head_max_len=head_max_len), ckpt_dir)
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()

    # Post-training temperature calibration on rank 0
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        try:
            records = calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16)
            fitted = fit_temperature_map(records)
            fitted_temps = fitted["temperature"]
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            fitted_temps, fitted = None, {"temperature_by_options": {}}

        out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len)
        out_cfg["fine_tuned"] = True
        out_cfg["model_name"] = "laya-asturcamin"
        if fitted_temps is not None:
            out_cfg["temperature"] = fitted_temps
            # This fit carries per-bucket overrides; inherited ones would shadow them.
            out_cfg.pop("temperature_by_options", None)
            if fitted["temperature_by_options"]:
                out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
        save_checkpoint(model, tok, out_cfg, output_dir)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs on both T4 GPUs in parallel (~4 to 6 minutes total).


In [ ]:
OUTPUT_DIR = "/kaggle/working/laya_asturcamin"
MODEL_DIR = model_dir

cmd = f"torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}"
print("Entrenando:", cmd)
!{cmd}


## 6. Medida sobre la validación

In [ ]:
# Medida sobre la validación (85 avisos que el modelo no ha visto al entrenar)
import json, time
import laya

agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
NIVELES = ["Baja", "Media", "Alta", "Crítica"]
aciertos = {"organismo": 0, "severidad": 0, "riesgo_vial": 0, "urgente": 0, "inspeccion": 0}
lat, n = [], 0
with open("/kaggle/working/datos/finetune_val.jsonl", encoding="utf-8") as fh:
    for linea in fh:
        row = json.loads(linea)
        state, questions, gold = (json.loads(row[k]) for k in ("state", "questions", "gold"))
        t0 = time.perf_counter()
        r = agent_ft.predict(state, questions)["answers"]
        lat.append((time.perf_counter() - t0) * 1000)
        n += 1
        aciertos["organismo"] += r["organismo"]["choice"] == gold["organismo"]["label"]
        p = r["severidad"].get("probabilities", {})
        nivel = int(max(p, key=p.get)) if p else int(round(r["severidad"]["score"]))
        aciertos["severidad"] += nivel == gold["severidad"]["label"]
        for k in ("riesgo_vial", "urgente", "inspeccion"):
            aciertos[k] += (r[k]["noul"] >= 0.5) == (gold[k]["label"] == "true")

print(f"=== Modelo afinado · validación ({n} avisos) ===")
for k, v in aciertos.items():
    print(f"{k:12s}{v}/{n} ({v / n:.0%})")
print(f"Latencia media en GPU: {sum(lat) / n:.0f} ms")


## 7. Empaquetar el modelo

In [ ]:
!cd /kaggle/working && zip -qr laya_asturcamin.zip laya_asturcamin && ls -lh laya_asturcamin.zip
print("Listo: descarga laya_asturcamin.zip desde el panel Output.")
